In [ ]:
from odin import PostgresWrapper
import os

In [ ]:
identity_db = PostgresWrapper(
    instance_connection_name=os.getenv("IDENTITY_DB_CONNECTION_NAME"),
    db_name=os.getenv("IDENTITY_DB_NAME"),
    user=os.getenv("IDENTITY_USER"),
    password=os.getenv("IDENTITY_PASSWORD"),
    ip_type="public",
)

In [ ]:
connected = identity_db.test_connection()

In [ ]:
print(identity_db.query_one("SELECT current_database(), current_user, current_schema(), version()"))
print(identity_db.query_one("SELECT current_user, rolsuper FROM pg_roles WHERE rolname = current_user"))

In [ ]:
tables = identity_db.query("SELECT tablename, tableowner FROM pg_tables WHERE schemaname='public' ORDER BY tablename")

tables = [t.get("tablename") for t in tables]

display(tables)

In [ ]:
tables_w_no_items = []

for table in tables:
    all_items = identity_db.get_all_from_table(table, limit=10)

    if len(all_items) == 0:
        tables_w_no_items.append(table)



In [ ]:
tables_w_items = [t for t in tables if t not in tables_w_no_items]

In [ ]:
tables_w_items

In [ ]:
table_lengths = {}

for table in tables_w_items:
    first_ten = identity_db.get_all_from_table(table)

    table_lengths[table] = len(first_ten)




In [ ]:
rmrcode = "RMRUSAD"

entity = identity_db.get_table_item_by_attribute("identity_crosswalk", "external_id", rmrcode)
relations = identity_db.get_table_item_by_attribute("identity_crosswalk", "entity_id", entity['entity_id'], first=False)
clickup = [c for c in relations if c.get("system_id") == 1]

client_id = None
if clickup:
    client_id = clickup[0].get("external_id")


client_id


In [ ]:
all_identities = identity_db.get_all_from_table("identity_crosswalk")

In [ ]:
relations = [e for e in all_identities if e.get("entity_id") == entity.get("entity_id")]

In [ ]:

# Query the identity database to look for clickup id's whos 
def clickup_id_from_rmrcode(db, rmrcode: str, rmr_system: str) -> str | None:
    return db.query_scalar(
        """
            SELECT cu.external_id
            FROM identity_crosswalk     src
            JOIN source_systems         src_sys ON src_sys.id = src.system_id
            JOIN identity_crosswalk     cu      ON cu.entity_id = src.entity_id
            JOIN source_systems         cu_sys  ON cu_sys.id = cu.system_id
            WHERE src.external_id   =   %s
            AND src_sys.code        =   %s
            AND cu_sys.code         =   'clickup'
        """,
        [rmrcode, rmr_system]
    )

client_id = clickup_id_from_rmrcode(identity_db, "RMROTREE", rmr_system="rmr")

print(client_id)


In [ ]:
identity_db.query("SELECT id, code, display_name, is_active FROM source_systems ORDER BY id")


In [ ]:

entity = identity_db.get_table_item_by_attribute("identity_crosswalk", "external_id", "RMROTREE")
entity

In [ ]:
clikcup_items = identity_db.query(
    """
        SELECT * FROM identity_crosswalk
        WHERE identity_crosswalk.external_id = %s
        LIMIT %s
    """,
    ["868k3tyep", 1]
)
clikcup_items

In [ ]:
clikcup_items = identity_db.query(
    """
        SELECT * FROM identity_crosswalk
        WHERE identity_crosswalk.entity_id = %s
        LIMIT %s
    """,
    ["f2f90208-9898-46c4-843a-23e7da3c7841", 10]
)
clikcup_items